In [ ]:
from pathlib import Path
import pandas as pd
import json
JSON_NAME = "until_2026-07-23_삼성전자.json"
BASE_DIR = Path.cwd().parent.parent
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PORCESSED_DIR = DATA_DIR / "processed"
JSON_PATH = RAW_DIR / JSON_NAME


In [ ]:
# 판다스 디스플레이 설정
pd.set_option("display.max_colwidth", None)
# JSON로드
records = []
with open(JSON_PATH, "r", encoding="utf-8-sig") as file:
    for line in file:
        line = line.strip()
        if not line:
            continue

        record = json.loads(line)

        cleaned_record = {
            key.strip(" :"): value
            for key, value in record.items()
        }

        records.append(cleaned_record)

raw_df = pd.DataFrame(records)
# 컬럼정제 (크롤링 전처리 요청사항)
raw_df.columns = raw_df.columns.str.strip(" :")
# 사용컬럼 선택
comments_df = raw_df[
    [
        "comment_id",
        "comment_text_raw",
        "board_stock_code",
        "created_at",
        "collected_at",
    ]
].copy()
raw_df.head()

In [ ]:
# 종목코드 정제 (크롤링 전처리 요청사항)
comments_df["board_stock_code"] = (
    comments_df["board_stock_code"]
    .astype("string")
    .str.strip()
    .str.replace(r"^A", "", regex=True)
)
# 본문정제(크롤링 전처리 요청사항)
comments_df["comment_text_raw"] = (
    comments_df["comment_text_raw"]
    .fillna("")
    .astype(str)
    .str.strip()
)
# 빈 문자열 제거 (크롤링 전처리 요청사항)
empty_mask = comments_df["comment_text_raw"].eq("")
comments_df = (
    comments_df[~empty_mask]
    .reset_index(drop=True)
)
# 줄바뀜 문자 보정
comments_df["comment_text_raw"] = (
    comments_df["comment_text_raw"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
# datetime 객체 변환
comments_df["created_at"] = pd.to_datetime(
    comments_df["created_at"],
    errors="coerce",
)
comments_df["collected_at"] = pd.to_datetime(
    comments_df["collected_at"],
    errors="coerce",
)
# 문자열 길이 컬럼 생성
comments_df["comment_length"] = (
    comments_df["comment_text_raw"].str.len()
)
# 댓글 생성일자 순으로 정렬
comments_df = comments_df.sort_values(
    ["board_stock_code", "created_at", "comment_id"]
)
# 중복제거
comments_df = (
    comments_df
    .sort_values("collected_at")
    .drop_duplicates(
        subset="comment_id",
        keep="last",
    )
    .reset_index(drop=True)
)



In [ ]:
print("전체 행:", len(comments_df))
print("고유 ID:", comments_df["comment_id"].nunique())
print("추가 중복 행:", comments_df["comment_id"].duplicated().sum())

duplicate_df = (
    comments_df[comments_df["comment_id"].duplicated(keep=False)]
    .sort_values(["comment_id", "collected_at"])
)

print("중복 ID 종류:", duplicate_df["comment_id"].nunique())
duplicate_df.head(20)

changed_ids = (
    duplicate_df.groupby("comment_id")["comment_text_raw"]
    .nunique()
    .loc[lambda x: x > 1]
    .index
)

duplicate_df[
    duplicate_df["comment_id"].isin(changed_ids)
]

In [ ]:
print("원본 댓글 수:", len(raw_df))
print("정제 후 댓글 수:", len(comments_df))

display(comments_df.head())
display(comments_df["comment_length"].describe())

In [ ]:
# 정제데이터 저장
comments_df.to_json(
    PORCESSED_DIR / "comments_cleaned.json",
    orient="records",
    force_ascii=False,
    date_format="iso",
    indent=2,
)

In [ ]:
# 시간 구간 생성
WINDOW = "1d"

comments_df["window_start"] = (
    comments_df["created_at"].dt.floor(WINDOW)
)
# 시간 구간별 댓글 집계
comment_window_df = (
    comments_df
    .groupby(
        ["board_stock_code", "window_start"],
        as_index=False,
    )
    .agg(
        comment_text=("comment_text_raw", " ".join),
        comment_count=("comment_id", "size"),
    )
)
comment_window_df.sort_values(by="comment_count", ascending=False)
comment_window_df[["comment_count", "window_start"]]